# Opdracht 5 · Decision trees & random forest
### Regressie-training · Antwoordmodel-versie

Beslisbomen vangen niet-lineaire verbanden, maar overfitten snel. Een random forest combineert veel bomen
en is veel robuuster. In deze opdracht ga je:

1. Een enkele **decision tree** trainen
2. Een **random forest** trainen
3. Beide vergelijken met het **lineaire model** uit opdracht 2
4. De **feature importance** van het bos bekijken
5. De **boomdiepte** variëren en het overfitting-effect zien

> **Dataset:** dezelfde Diabetes-dataset als in opdracht 1 en 2, zodat je de modellen direct kunt vergelijken.

> **Hoe werkt dit notebook?** De meeste cellen zijn ingevuld; op de plekken met **`# TODO`** vul jij iets in.


## 0. Voorbereiding
Imports en data laden/splitsen (zelfde opzet als opdracht 2). Gewoon uitvoeren.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

plt.rcParams["figure.figsize"] = (7, 4)
print("Bibliotheken geladen.")

In [ ]:
data = load_diabetes(as_frame=True)
df = data.frame
y = df["target"]
X = df.drop(columns=["target"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Referentie: het lineaire model uit opdracht 2
lineair = LinearRegression().fit(X_train, y_train)
r2_lineair = r2_score(y_test, lineair.predict(X_test))
print(f"R² lineair model (uit opdracht 2): {r2_lineair:.3f}")

## 1. Een enkele decision tree
Een beslisboom splitst de data stapsgewijs op. Zonder beperking groeit hij door tot hij de
trainingsdata vrijwel uit het hoofd kent — met overfitting als gevolg.

### TODO 1 — Train een decision tree
Maak een `DecisionTreeRegressor` met `random_state=42`, train op de train-set en bereken de R² op de test-set.

> Tip: `DecisionTreeRegressor(random_state=42)`, dan `.fit(X_train, y_train)`.
> Voor de R²: `r2_score(y_test, model.predict(X_test))`.

In [ ]:
# ANTWOORD
tree = DecisionTreeRegressor(random_state=42)
tree.fit(X_train, y_train)
r2_tree = r2_score(y_test, tree.predict(X_test))
print(f"R² decision tree: {r2_tree:.3f}")

> Vergelijk: hoe verhoudt de R² van deze enkele boom zich tot het lineaire model?
> Een lage of zelfs lagere score wijst op overfitting — daar komen we bij stap 5 op terug.

## 2. Een random forest
Een random forest traint veel bomen op willekeurige steekproeven en middelt hun voorspellingen.
Dat dempt de overfitting van losse bomen.

### TODO 2 — Train een random forest
Maak een `RandomForestRegressor` met `n_estimators=100` en `random_state=42`, train op de train-set
en bereken de R² op de test-set.

> Tip: `RandomForestRegressor(n_estimators=100, random_state=42)`.

In [ ]:
# ANTWOORD
forest = RandomForestRegressor(n_estimators=100, random_state=42)
forest.fit(X_train, y_train)
r2_forest = r2_score(y_test, forest.predict(X_test))
print(f"R² random forest: {r2_forest:.3f}")

## 3. De drie modellen vergelijken
Zet de R²-scores naast elkaar (al ingevuld).

In [ ]:
# Vergelijking van de drie modellen (al ingevuld)
scores = pd.Series({
    "Lineair": r2_lineair,
    "Decision tree": r2_tree,
    "Random forest": r2_forest,
})
print(scores.round(3))

scores.plot(kind="bar", color=["#9199B9", "#D97733", "#4473C5"])
plt.ylabel("R² op de test-set")
plt.title("Modelvergelijking")
plt.xticks(rotation=0)
plt.axhline(0, color="black", linewidth=0.8)
plt.show()

> **Bespreek:** welk model wint? Vaak presteert het random forest vergelijkbaar met of beter dan het
> lineaire model, terwijl de losse boom achterblijft door overfitting.

## 4. Feature importance
Een random forest kan aangeven welke features het meest bijdragen aan de voorspelling.

### TODO 3 — Bekijk de feature importance
Maak een `pandas.Series` van `forest.feature_importances_` met `X.columns` als index,
en sorteer aflopend.

> Tip: `pd.Series(forest.feature_importances_, index=X.columns).sort_values(ascending=False)`.

In [ ]:
# ANTWOORD
importance = pd.Series(forest.feature_importances_, index=X.columns)
importance = importance.sort_values(ascending=False)
print(importance.round(3))

importance.plot(kind="barh", color="#4473C5")
plt.gca().invert_yaxis()
plt.xlabel("Belang")
plt.title("Feature importance (random forest)")
plt.show()

> **Herken je deze features?** Vergelijk de belangrijkste features met de sterkste coëfficiënten
> die je in opdracht 2 vond. Komen ze overeen?

## 5. Het effect van boomdiepte
De diepte van een boom bepaalt hoe complex hij mag worden. Te diep = overfitting. We trainen bomen van
verschillende diepte en vergelijken de R² op **train** en **test**.

### TODO 4 — Vul de boomdiepte-lijst aan
Vul de lijst `dieptes` aan met enkele waarden om te testen. Begin klein en loop op,
bijvoorbeeld 1, 2, 3, 5, 10 (en `None` = onbeperkt is al toegevoegd).

> Tip: de lijst is een gewone Python-lijst, bijv. `[1, 2, 3, 5, 10, None]`.

In [ ]:
# ANTWOORD
dieptes = [1, 2, 3, 5, 10, None]

for d in dieptes:
    t = DecisionTreeRegressor(max_depth=d, random_state=42).fit(X_train, y_train)
    r2_train = t.score(X_train, y_train)
    r2_test = r2_score(y_test, t.predict(X_test))
    print(f"diepte {str(d):>4}  ->  train R² {r2_train:.3f}  |  test R² {r2_test:.3f}")

> **Let op het patroon:** naarmate de boom dieper wordt, stijgt de **train**-score richting 1,0,
> maar de **test**-score zakt weer in. Dat gat tussen train en test is precies wat **overfitting** is.

## 6. Bespreking
Bespreek met je buur:

- Welk model gaf de beste test-R²? En wat kostte dat aan interpreteerbaarheid?
- Waarom doet de enkele, onbeperkte boom het zo slecht op de test-set, terwijl hij op train bijna perfect is?
- Bij welke diepte generaliseert de boom het best? Hoe zie je dat in de train/test-scores?
- Komen de belangrijkste features van het bos overeen met de sterke coëfficiënten uit opdracht 2?

---
### Toelichting voor de trainer
- Verwacht ongeveer: **lineair R² ≈ 0,45**, **enkele boom R² ≈ 0,06** (sterk overfit),
  **random forest R² ≈ 0,44** (herstelt richting het lineaire model).
- **Feature importance**: `bmi` (≈ 0,36) en `s5` (≈ 0,23) bovenaan — sluit mooi aan op de sterke
  coëfficiënten uit opdracht 2.
- **Diepte-sweep** is het kernpunt: bij `None` haalt de boom train R² = 1,0 maar test R² ≈ 0,06
  (schoolvoorbeeld van overfitting); rond diepte 3-5 generaliseert hij het best (test R² ≈ 0,33).
- Boodschap: complexer is niet automatisch beter; het random forest geeft de robuustheid zonder
  handmatig de diepte te moeten afstellen, ten koste van interpreteerbaarheid.